# Week 7: CNNs and Visual Representations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/07/Week_07_CNNs_Visual_Representations.ipynb)

**Course:** Neural Architectures and Representation Learning (Master level)


## Learning goals

By the end of this session, you should be able to:

- Explain convolution as a local pattern-detection operation.
- Train a small CNN for image classification.
- Inspect learned filters and feature maps.
- Explain why deeper CNN layers can behave like reusable visual representations.
- Use a pretrained CNN as a frozen feature extractor.
- Train a small head on top of a latent representation for a new visual task.

**Course habit:** change one thing -> run -> observe -> explain.

**New representation habit:** inspect what the model carries forward internally, not only the final prediction.


---

## Environment

**Dependencies:** `torch`, `torchvision`, `numpy`, `matplotlib`. CPU is enough for the core path.

### Local (uv)

From the repo root:

```bash
uv sync
uv run jupyter notebook weeks/07/Week_07_CNNs_Visual_Representations.ipynb
```

### Colab

1. Open the notebook via the badge above.
2. Runtime -> Change runtime type -> CPU is fine.
3. Run the setup cell below.

The CIFAR/pretrained section may download CIFAR-10 and pretrained weights. If that is slow, use the MNIST CNN section as the guaranteed core path and treat the pretrained section as a demo.


In [ ]:
import math
import random
import colorsys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms, models

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

print("torch:", torch.__version__)
print("numpy:", np.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(7)


---

## 1. Kernel intuition

Before CNNs, open this interactive page for a few minutes:

[Setosa image kernels](https://setosa.io/ev/image-kernels/)

Try edge detection, blur, and sharpen.

**Mental model:** a kernel is a small local rule that slides across an image. It responds strongly when the local pixels match the pattern encoded in the kernel.

**Transition:** CNNs learn useful kernels from data instead of us hand-picking them.

**Pause and predict**

1. What kind of image region should an edge filter respond to?
2. Why does a blur filter remove detail?
3. If a CNN learns its filters, what decides whether a filter becomes useful?


---

## 2. Our own tiny CNN on MNIST

MNIST is simple, grayscale, and small. That makes it a good first CNN playground.

We will train a tiny CNN end-to-end:

`Conv -> ReLU -> Pool -> Conv -> ReLU -> Pool -> Flatten -> Linear`

The important part is not the final accuracy. The important part is seeing how pixels become feature maps and then a compact representation.


In [ ]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])


def make_digits_fallback():
    """Fallback for flaky MNIST downloads: sklearn digits, upsampled to 28x28."""
    from sklearn.datasets import load_digits

    digits = load_digits()
    X = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
    X = F.interpolate(X, size=(28, 28), mode="bilinear", align_corners=False)
    X = (X - 0.1307) / 0.3081
    y = torch.tensor(digits.target, dtype=torch.long)
    split = int(0.8 * len(X))
    return TensorDataset(X[:split], y[:split]), TensorDataset(X[split:], y[split:])

try:
    mnist_train_full = datasets.MNIST(DATA_DIR, train=True, download=True, transform=mnist_transform)
    mnist_test_full = datasets.MNIST(DATA_DIR, train=False, download=True, transform=mnist_transform)
    train_indices = list(range(min(6000, len(mnist_train_full))))
    test_indices = list(range(min(1200, len(mnist_test_full))))
    mnist_train = Subset(mnist_train_full, train_indices)
    mnist_test = Subset(mnist_test_full, test_indices)
    print("Loaded MNIST.")
except Exception as exc:
    print("MNIST download/load failed; using sklearn digits fallback.")
    print("Reason:", repr(exc))
    mnist_train, mnist_test = make_digits_fallback()

train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=256, shuffle=False)

images, labels = next(iter(train_loader))
print("batch image shape:", tuple(images.shape))
print("batch label shape:", tuple(labels.shape))


In [ ]:
def show_mnist_batch(images, labels, n=12):
    plt.figure(figsize=(10, 3))
    for i in range(n):
        plt.subplot(2, n // 2, i + 1)
        img = images[i, 0].detach().cpu().numpy()
        plt.imshow(img, cmap="gray")
        plt.title(str(int(labels[i])))
        plt.axis("off")
    plt.suptitle("MNIST samples")
    plt.tight_layout()
    plt.show()

show_mnist_batch(images, labels)


In [ ]:
class TinyMNISTCNN(nn.Module):
    def __init__(self, channels1=8, channels2=16, kernel_size=3, representation_dim=32):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv2d(1, channels1, kernel_size=kernel_size, padding=padding)
        self.conv2 = nn.Conv2d(channels1, channels2, kernel_size=kernel_size, padding=padding)
        self.pool = nn.MaxPool2d(2)
        self.representation = nn.Linear(channels2 * 7 * 7, representation_dim)
        self.classifier = nn.Linear(representation_dim, 10)

    def forward_features(self, x):
        z1 = self.conv1(x)
        a1 = F.relu(z1)
        p1 = self.pool(a1)
        z2 = self.conv2(p1)
        a2 = F.relu(z2)
        p2 = self.pool(a2)
        flat = p2.flatten(1)
        rep = F.relu(self.representation(flat))
        return rep, {"conv1": z1, "act1": a1, "pool1": p1, "conv2": z2, "act2": a2, "pool2": p2}

    def forward(self, x):
        rep, _ = self.forward_features(x)
        return self.classifier(rep)

model = TinyMNISTCNN().to(device)
print(model)

with torch.no_grad():
    logits = model(images[:4].to(device))
print("logit shape:", tuple(logits.shape))


### Shape check

CNN layers change the tensor shape:

- `N x 1 x 28 x 28`: batch of grayscale images
- after `conv1`: multiple feature maps, still spatial
- after pooling: smaller spatial maps
- after flattening: a vector representation
- after classifier: 10 logits, one per digit

**Pause and predict:** Why might spatial feature maps be better than flattening the image immediately?


In [ ]:
def accuracy_from_logits(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = X.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_count += batch_size
    return total_loss / total_count, total_correct / total_count


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = loss_fn(logits, y)
        batch_size = X.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_count += batch_size
    return total_loss / total_count, total_correct / total_count


def train_mnist_cnn(channels1=8, channels2=16, kernel_size=3, epochs=2, lr=1e-3, seed=7):
    set_seed(seed)
    model = TinyMNISTCNN(channels1=channels1, channels2=channels2, kernel_size=kernel_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn)
        test_loss, test_acc = evaluate(model, test_loader, loss_fn)
        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)
        print(f"epoch {epoch+1:02d} | train acc {train_acc:.3f} | test acc {test_acc:.3f} | test loss {test_loss:.3f}")
    return model, history


---

## 3. Full tiny CNN training

Run the full training once. It is intentionally small, because the classroom goal is understanding the representation flow, not leaderboard performance.


In [ ]:
mnist_model, mnist_history = train_mnist_cnn(epochs=2, lr=1e-3)


In [ ]:
def plot_training_history(history, title="Training history"):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, history["train_loss"], marker="o", label="train")
    axes[0].plot(epochs, history["test_loss"], marker="o", label="test")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[1].plot(epochs, history["train_acc"], marker="o", label="train")
    axes[1].plot(epochs, history["test_acc"], marker="o", label="test")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_training_history(mnist_history, "Tiny CNN on MNIST")


In [ ]:
@torch.no_grad()
def show_predictions(model, loader, n=12):
    model.eval()
    X, y = next(iter(loader))
    logits = model(X.to(device))
    preds = logits.argmax(dim=1).cpu()
    plt.figure(figsize=(10, 3))
    for i in range(n):
        plt.subplot(2, n // 2, i + 1)
        plt.imshow(X[i, 0].numpy(), cmap="gray")
        color = "green" if preds[i].item() == y[i].item() else "red"
        plt.title(f"p={preds[i].item()} / y={y[i].item()}", color=color)
        plt.axis("off")
    plt.suptitle("Predictions from the tiny CNN")
    plt.tight_layout()
    plt.show()

show_predictions(mnist_model, test_loader)


### Inspect learned filters

The first convolutional layer has learned small local filters. They are not as clean as hand-designed kernels, but they already act like local detectors.


In [ ]:
def show_first_layer_filters(model):
    weights = model.conv1.weight.detach().cpu()  # [out_channels, 1, k, k]
    n = weights.shape[0]
    cols = min(8, n)
    rows = math.ceil(n / cols)
    plt.figure(figsize=(1.6 * cols, 1.6 * rows))
    for i in range(n):
        plt.subplot(rows, cols, i + 1)
        filt = weights[i, 0]
        vmax = float(filt.abs().max())
        plt.imshow(filt, cmap="coolwarm", vmin=-vmax, vmax=vmax)
        plt.title(f"filter {i}")
        plt.axis("off")
    plt.suptitle("First-layer learned filters")
    plt.tight_layout()
    plt.show()

show_first_layer_filters(mnist_model)


### Inspect feature maps

A feature map shows where one learned filter responds strongly in the image.

Early feature maps often still look close to pixels and edges. Later feature maps are more abstract and harder to interpret directly.


In [ ]:
@torch.no_grad()
def show_feature_maps(model, image, layer_name="act1", max_maps=8):
    model.eval()
    image = image.unsqueeze(0).to(device)
    _, features = model.forward_features(image)
    maps = features[layer_name][0].detach().cpu()
    n = min(max_maps, maps.shape[0])

    plt.figure(figsize=(2.0 * (n + 1), 2.2))
    plt.subplot(1, n + 1, 1)
    plt.imshow(image[0, 0].detach().cpu().numpy(), cmap="gray")
    plt.title("input")
    plt.axis("off")
    for i in range(n):
        plt.subplot(1, n + 1, i + 2)
        plt.imshow(maps[i], cmap="viridis")
        plt.title(f"map {i}")
        plt.axis("off")
    plt.suptitle(f"Feature maps from {layer_name}")
    plt.tight_layout()
    plt.show()

sample_image, sample_label = mnist_test[17]
print("label:", sample_label)
show_feature_maps(mnist_model, sample_image, "act1", max_maps=8)
show_feature_maps(mnist_model, sample_image, "act2", max_maps=8)


**Pause and reflect**

1. Which feature maps still look visually close to the digit?
2. Which maps are harder to interpret directly?
3. Why might a later representation be useful even when it is less human-readable?


---

## 4. Coding block 1: change the tiny CNN

**Goal:** edit the architecture, train briefly, and inspect the effect.

**Core path**

1. Change `channels1`, `channels2`, or `kernel_size`.
2. Re-run a short training.
3. Compare accuracy and feature maps.
4. Explain what changed.

Keep the run short. The point is comparison, not squeezing out the last percent of accuracy.


In [ ]:
# TODO: edit one or two values, then re-run.
student_config = {
    "channels1": 8,
    "channels2": 16,
    "kernel_size": 3,  # try 3 or 5
    "epochs": 1,
    "lr": 1e-3,
}

student_model, student_history = train_mnist_cnn(**student_config)
plot_training_history(student_history, "Student CNN experiment")
show_first_layer_filters(student_model)
show_feature_maps(student_model, sample_image, "act1", max_maps=8)


### Experiment report prompt

Write 3-5 sentences:

1. What did you change?
2. What happened to accuracy or loss?
3. What happened to filters or feature maps?
4. Did the change improve the representation, or only the metric?


---

## 5. External CNN visualizers

Now that you have seen the code, use external visualizers to make the layer flow more tangible.

1. [Adam Harley MNIST CNN visualizer](https://adamharley.com/nn_vis/cnn/2d.html)
2. [MNIST/CNN visualization page](https://www.brilliantwavetech.com/cnn-visualization.html)
3. [CNN Explainer](https://poloclub.github.io/cnn-explainer/)

**Use them with a specific question:** Where do local pixel patterns become feature maps, and where do feature maps become a compact representation for the classifier?

**Quick activity:** draw a digit that is ambiguous, then watch which activations and output probabilities change.


---

## 6. From CNNs to reusable representations

A trained CNN does not only output a class label. Inside the model, it builds a latent representation of the image.

We can reuse that representation for a new task by freezing the CNN backbone and training a small head.

Today we use a simple color task on CIFAR-10:

1. Take a color image.
2. Compute its average RGB color.
3. Convert that color to hue and saturation.
4. Put hue and saturation on a disk.
5. Train a small head to predict that disk point from CNN features.

This task is deliberately simple. It lets us ask whether a visual representation preserves useful color information.


In [ ]:
def rgb_to_disk_coordinates(rgb_array):
    """Convert average RGB values in [0, 1] to hue/saturation disk coordinates."""
    coords = []
    hsv_values = []
    for r, g, b in rgb_array:
        h, s, v = colorsys.rgb_to_hsv(float(r), float(g), float(b))
        angle = 2 * math.pi * h
        coords.append([s * math.cos(angle), s * math.sin(angle)])
        hsv_values.append([h, s, v])
    return np.array(coords, dtype=np.float32), np.array(hsv_values, dtype=np.float32)


def disk_to_rgb(xy):
    """Map disk coordinates back to vivid RGB colors for plotting."""
    colors = []
    for x, y in xy:
        s = min(1.0, math.sqrt(float(x * x + y * y)))
        h = (math.atan2(float(y), float(x)) / (2 * math.pi)) % 1.0
        colors.append(colorsys.hsv_to_rgb(h, s, 0.95))
    return np.array(colors)


def plot_color_disk(coords, title, images=None, sample_count=0):
    colors = disk_to_rgb(coords)
    plt.figure(figsize=(6, 6))
    circle = plt.Circle((0, 0), 1.0, color="black", fill=False, linewidth=1.0, alpha=0.4)
    ax = plt.gca()
    ax.add_patch(circle)
    plt.scatter(coords[:, 0], coords[:, 1], c=colors, s=28, alpha=0.8, edgecolor="black", linewidth=0.2)
    plt.axhline(0, color="black", linewidth=0.5, alpha=0.4)
    plt.axvline(0, color="black", linewidth=0.5, alpha=0.4)
    plt.xlim(-1.05, 1.05)
    plt.ylim(-1.05, 1.05)
    plt.gca().set_aspect("equal")
    plt.xlabel("saturation * cos(hue)")
    plt.ylabel("saturation * sin(hue)")
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
cifar_base_transform = transforms.Compose([
    transforms.ToTensor(),
])

try:
    cifar_train_full = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=cifar_base_transform)
    cifar_test_full = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=cifar_base_transform)
    class_names = cifar_train_full.classes
    print("Loaded CIFAR-10.")
except Exception as exc:
    print("CIFAR-10 download/load failed; using torchvision FakeData fallback.")
    print("Reason:", repr(exc))
    cifar_train_full = datasets.FakeData(size=1024, image_size=(3, 32, 32), num_classes=10, transform=cifar_base_transform)
    cifar_test_full = datasets.FakeData(size=512, image_size=(3, 32, 32), num_classes=10, transform=cifar_base_transform)
    class_names = [f"class_{i}" for i in range(10)]

# Keep the representation demo small. Feature extraction is the expensive part.
cifar_train = Subset(cifar_train_full, list(range(min(512, len(cifar_train_full)))))
cifar_test = Subset(cifar_test_full, list(range(min(256, len(cifar_test_full)))))

cifar_train_loader = DataLoader(cifar_train, batch_size=64, shuffle=False)
cifar_test_loader = DataLoader(cifar_test, batch_size=64, shuffle=False)

print("CIFAR/fallback classes:", class_names)


In [ ]:
def show_cifar_samples(dataset, n=12):
    plt.figure(figsize=(10, 3))
    for i in range(n):
        img, label = dataset[i]
        plt.subplot(2, n // 2, i + 1)
        plt.imshow(img.permute(1, 2, 0).numpy())
        plt.title(class_names[label])
        plt.axis("off")
    plt.suptitle("CIFAR-10 color samples")
    plt.tight_layout()
    plt.show()

show_cifar_samples(cifar_train)


In [ ]:
@torch.no_grad()
def compute_color_targets(loader):
    all_coords = []
    all_hsv = []
    all_images = []
    all_labels = []
    for X, y in loader:
        avg_rgb = X.mean(dim=(2, 3)).numpy()
        coords, hsv = rgb_to_disk_coordinates(avg_rgb)
        all_coords.append(coords)
        all_hsv.append(hsv)
        all_images.append(X)
        all_labels.append(y)
    return (
        torch.tensor(np.concatenate(all_coords), dtype=torch.float32),
        np.concatenate(all_hsv),
        torch.cat(all_images),
        torch.cat(all_labels),
    )

train_disk, train_hsv, train_images_raw, train_labels = compute_color_targets(cifar_train_loader)
test_disk, test_hsv, test_images_raw, test_labels = compute_color_targets(cifar_test_loader)

plot_color_disk(train_disk.numpy(), "True average hue/saturation disk: CIFAR train subset")


### Why disk coordinates?

Hue wraps around: red is near both 0 and 1 on the hue scale. Predicting raw hue makes red near 0 and red near 1 look far apart numerically.

Disk coordinates avoid this:

```text
x = saturation * cos(hue)
y = saturation * sin(hue)
```

Angle represents hue. Radius represents saturation.


---

## 7. Pretrained CNN features + color disk head

We now freeze a CNN backbone and train only a small regression head.

**Representation learning point:** the backbone produces a latent vector. The head learns a new task from that vector.

If pretrained weights are unavailable, the code falls back to an untrained ResNet. That fallback still demonstrates the mechanics, but the representation will be less meaningful.


In [ ]:
def build_resnet_feature_extractor(pretrained=True):
    try:
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)
        if weights is not None:
            print("Loaded ImageNet-pretrained ResNet18 weights.")
        transform = weights.transforms() if weights is not None else transforms.Compose([
            transforms.Resize((96, 96)),
            transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
        ])
    except Exception as exc:
        print("Could not load pretrained weights; using random ResNet18.")
        print("Reason:", repr(exc))
        backbone = models.resnet18(weights=None)
        transform = transforms.Compose([
            transforms.Resize((96, 96)),
            transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
        ])

    feature_dim = backbone.fc.in_features
    backbone.fc = nn.Identity()
    backbone.eval().to(device)
    for p in backbone.parameters():
        p.requires_grad = False
    return backbone, transform, feature_dim

backbone, backbone_transform, feature_dim = build_resnet_feature_extractor(pretrained=True)
print("feature dimension:", feature_dim)


In [ ]:
@torch.no_grad()
def extract_features(loader, backbone, transform):
    features = []
    labels = []
    coords = []
    raw_images = []
    for X, y in loader:
        # X is already [0, 1]. ResNet weights transform can accept batched tensors.
        X_for_backbone = transform(X).to(device)
        feats = backbone(X_for_backbone).cpu()
        avg_rgb = X.mean(dim=(2, 3)).numpy()
        disk_coords, _ = rgb_to_disk_coordinates(avg_rgb)
        features.append(feats)
        labels.append(y)
        coords.append(torch.tensor(disk_coords, dtype=torch.float32))
        raw_images.append(X)
    return torch.cat(features), torch.cat(coords), torch.cat(labels), torch.cat(raw_images)

train_features, train_targets, train_cls, train_imgs = extract_features(cifar_train_loader, backbone, backbone_transform)
test_features, test_targets, test_cls, test_imgs = extract_features(cifar_test_loader, backbone, backbone_transform)

print("train features:", tuple(train_features.shape))
print("train color targets:", tuple(train_targets.shape))


In [ ]:
class ColorDiskHead(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        return self.net(x)


def train_color_head(train_features, train_targets, test_features, test_targets, hidden_dim=64, lr=1e-3, epochs=80, seed=7):
    set_seed(seed)
    head = ColorDiskHead(train_features.shape[1], hidden_dim=hidden_dim).to(device)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    X_train = train_features.to(device)
    y_train = train_targets.to(device)
    X_test = test_features.to(device)
    y_test = test_targets.to(device)
    history = {"train_mse": [], "test_mse": []}
    for epoch in range(epochs):
        head.train()
        optimizer.zero_grad()
        pred = head(X_train)
        loss = loss_fn(pred, y_train)
        loss.backward()
        optimizer.step()

        head.eval()
        with torch.no_grad():
            test_loss = loss_fn(head(X_test), y_test)
        history["train_mse"].append(float(loss.item()))
        history["test_mse"].append(float(test_loss.item()))
    return head, history

color_head, color_history = train_color_head(train_features, train_targets, test_features, test_targets, epochs=80)
print("final train mse:", color_history["train_mse"][-1])
print("final test mse: ", color_history["test_mse"][-1])


In [ ]:
def plot_mse_history(history, title="Color disk head training"):
    plt.figure(figsize=(6, 4))
    plt.plot(history["train_mse"], label="train")
    plt.plot(history["test_mse"], label="test")
    plt.xlabel("epoch")
    plt.ylabel("MSE")
    plt.yscale("log")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_mse_history(color_history)


In [ ]:
@torch.no_grad()
def predict_disk(head, features):
    head.eval()
    return head(features.to(device)).cpu().numpy()

pred_disk = predict_disk(color_head, test_features)
true_disk = test_targets.numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, coords, title in [
    (axes[0], true_disk, "True hue/saturation disk"),
    (axes[1], pred_disk, "Predicted hue/saturation disk"),
]:
    colors = disk_to_rgb(coords)
    ax.add_patch(plt.Circle((0, 0), 1.0, color="black", fill=False, linewidth=1.0, alpha=0.4))
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=28, alpha=0.8, edgecolor="black", linewidth=0.2)
    ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
    ax.axvline(0, color="black", linewidth=0.5, alpha=0.4)
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_aspect("equal")
    ax.set_title(title)
plt.tight_layout()
plt.show()


In [ ]:
def show_color_prediction_examples(images, true_xy, pred_xy, labels, n=12):
    errors = np.linalg.norm(true_xy - pred_xy, axis=1)
    order = np.argsort(errors)
    picks = np.concatenate([order[: n // 2], order[-(n // 2):]])
    plt.figure(figsize=(12, 4))
    for j, idx in enumerate(picks):
        img = images[idx].permute(1, 2, 0).numpy()
        plt.subplot(2, n // 2, j + 1)
        plt.imshow(img)
        plt.title(f"{class_names[int(labels[idx])] }\nerr={errors[idx]:.2f}", fontsize=9)
        plt.axis("off")
    plt.suptitle("Easy examples first, high-error examples second")
    plt.tight_layout()
    plt.show()

show_color_prediction_examples(test_imgs, true_disk, pred_disk, test_cls, n=12)


**Pause and reflect**

1. Which images are easy for the head?
2. Which images have ambiguous average color?
3. Does the representation seem to preserve color directly, indirectly, or imperfectly?
4. Why is this still transfer learning even though the target is not a class label?


---

## 8. Coding block 2: representation-head experiments

**Goal:** keep the CNN representation fixed and experiment with the small head.

**Core path**

1. Change `hidden_dim`, `lr`, or `epochs`.
2. Train a new head.
3. Compare true vs predicted disk points.
4. Inspect low-error and high-error samples.
5. Explain what the representation seems to preserve.


In [ ]:
# TODO: edit these values and re-run.
head_config = {
    "hidden_dim": 32,
    "lr": 1e-3,
    "epochs": 60,
}

student_head, student_color_history = train_color_head(
    train_features,
    train_targets,
    test_features,
    test_targets,
    **head_config,
)
plot_mse_history(student_color_history, "Student color-disk head")
student_pred_disk = predict_disk(student_head, test_features)
show_color_prediction_examples(test_imgs, true_disk, student_pred_disk, test_cls, n=12)


### Representation report prompt

Write 4-6 sentences:

1. What configuration did you try?
2. Did the head predict the color disk well?
3. Which examples had high error?
4. What does this suggest about the frozen CNN representation?
5. What would you try next: a different layer, a bigger head, or partial fine-tuning?


---

## 9. Assignment 2 preview

Assignment 2 will ask you to use CNN representations, not only CNN predictions.

Minimum evidence will likely include:

- a trained or reused CNN representation
- a small task-specific head
- a training curve or comparison plot
- at least one feature-map, activation, or latent-space visualization
- a short explanation of what was reused and what was learned

The main question is not "did the model get a high score?" The main question is: **what visual information did the representation make available?**


---

## Wrap-up: takeaways

1. A convolution kernel is a local pattern detector.
2. CNNs learn many kernels and produce many feature maps.
3. Early feature maps are often easier to interpret visually than later representations.
4. Later CNN representations can be reused by small heads for new tasks.
5. Representation learning means we care about the internal features, not only the final output.


---

## Homework / Post-class Extensions

Optional unless assigned.

| Idea | What to try |
|------|-------------|
| MLP vs CNN | Train a small MLP on MNIST and compare accuracy/feature logic |
| More feature maps | Visualize activations for several digits and compare responses |
| Earlier vs later representation | Train the color head from a different ResNet layer |
| Brightness target | Predict value/brightness in addition to hue and saturation |
| Partial fine-tuning | Unfreeze the last ResNet block and compare with frozen features |
| Hover visualization | Build an interactive disk where hovering over a point shows the image |


In [ ]:
# Extension scaffold: hover visualization idea.
# Keep this optional because browser behavior differs between local Jupyter and Colab.
# Suggested route: use matplotlib event callbacks locally or Plotly/Bokeh in Colab.

# TODO:
# 1. Create a scatter plot of true_disk or pred_disk.
# 2. On mouse hover, find the nearest point.
# 3. Display the corresponding CIFAR image next to the plot.
pass
